# Shor order finding for 15

Run the compiled period-finding core used in the small N=15 Shor demonstration and recover non-trivial factors classically.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

The quantum core of Shor's algorithm estimates a modular order; classical post-processing converts phase candidates into factors.

In [2]:
from fractions import Fraction
from math import gcd
from qiskit.circuit.library import QFT

# For a=2 mod 15, the modular order is r=4. The phase-register state below is
# the exact compiled order-finding core after modular exponentiation.
counting = 4
circuit = QuantumCircuit(counting)
circuit.h(range(counting))
for wire in range(counting):
    circuit.p(2 * np.pi * (2 ** wire) / 4, wire)
circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))

def probabilities_from_qiskit():
    return Statevector.from_instruction(circuit).probabilities()

/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_9309/3705106520.py:12: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))


## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(probabilities_from_qiskit)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def probabilities_from_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return np.abs(np.asarray(state)) ** 2

candidate, mettleq_ms, _ = benchmark(probabilities_from_mettleq)
error = max_abs_error(reference, candidate)
phase_integer = int(np.argmax(candidate))
phase_fraction = Fraction(phase_integer, 2 ** counting).limit_denominator(15)
order = phase_fraction.denominator
factors = sorted({gcd(pow(2, order // 2) - 1, 15), gcd(pow(2, order // 2) + 1, 15)})
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The compiled phase distribution must agree and the recovered factors of 15 must be exactly 3 and 5.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/09_shor_order_finding.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="compiled phase distribution atol=2e-6 and factors 3,5",
    passed=error <= 2e-6 and factors == [3, 5],
    exact_match=factors == [3, 5],
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "phase_integer": phase_integer, "order": order, "factors": factors},
    notes="This is the small compiled order-finding core, not a scalable modular-arithmetic implementation.",
)


Comparison summary
------------------
Correctness contract: PASS — compiled phase distribution atol=2e-6 and factors 3,5
SDK reference median: 0.496 ms
MettleQ median:       0.918 ms
Timing interpretation: the SDK reference was 1.848x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes
Note: This is the small compiled order-finding core, not a scalable modular-arithmetic implementation.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "compiled phase distribution atol=2e-6 and factors 3,5", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"factors": [3, 5], "max_probability_error": 2.3841857776929487e-07, "order": 4, "phase_integer": 4}, "mettleq_median_ms": 0.9175839950330555, "notebook": "qiskit/09_shor_order_finding.ipynb", "notes": "This is the small compiled order-finding core, not a scalable modular-arithmetic implementation.", "passed": true, "python": "3.13.2", "reference_median_ms

## What should you conclude?

This is a small pedagogical order-finding instance, not a claim that useful cryptographic factoring is locally tractable.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.